[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C42_Learning_Theory_Course/05_generalization_deep/05_讲解.ipynb)

# 05 · 深度泛化（随机标签 · margin · PAC-Bayes · 双下降）

目标：复现 **Zhang 随机标签可拟合**、计算 **归一化 margin 分布**、数值计算 **非平凡 PAC-Bayes 界**（反演 KL）、复现 **双下降** 的最小范数机制。

路线：随机标签可拟合(经典界失效) → 真实 vs 随机标签的 margin 分布 → PAC-Bayes 反演 KL 界 + Pinsker 对比 → 双下降(最小范数范数爆炸) → ✏️ 练习 → 📖 答案 → 🧪 真实数据胶囊。

> 全课收口：把模块 04 的隐式偏置*量化*成数据依赖的泛化界。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
print('numpy', np.__version__)

## 1 · Zhang 难题：过参数化拟合随机标签，且有效容量 ≪ 最坏容量

$n<d$ 的线性模型能拟合**任意**标签（含随机）。我们在**同一模型类**上对比：拟合随机标签（零训练误差但测试=随机猜）vs 拟合真实结构标签（零训练误差且测试好）——直接量出『最坏容量大、有效容量小』。

In [ ]:
n, d = 40, 80                       # 过参数化 d > n
# 真实结构: 标签由一个稀疏 teacher 生成(可泛化)
w_teacher = np.zeros(d); w_teacher[:3] = [2.0, -1.5, 1.0]
def gen(n, seed, random_labels=False):
    g = np.random.default_rng(seed); X = g.standard_normal((n, d))
    y = np.sign(X @ w_teacher) if not random_labels else g.choice([-1.,1.], n)
    return X, y
def fit_interp(X, y):
    return X.T @ np.linalg.solve(X @ X.T, y)    # 最小范数插值
# 拟合随机标签
Xr, yr = gen(n, 1, random_labels=True); wr = fit_interp(Xr, yr)
train_err_rand = np.mean(np.sign(Xr@wr) != yr)
# 拟合真实标签
Xs, ys = gen(n, 2, random_labels=False); ws = fit_interp(Xs, ys)
train_err_real = np.mean(np.sign(Xs@ws) != ys)
print(f'随机标签: 训练误差={train_err_rand:.3f}  (零!)')
print(f'真实标签: 训练误差={train_err_real:.3f}  (也零)')
assert train_err_rand < 1e-9 and train_err_real < 1e-9, '过参数化下两者都能零训练误差'
print('✅ 同一模型类能拟合随机标签 -> 最坏容量巨大 -> 经典容量界 vacuous。')

In [ ]:
# 测试误差对比: 随机标签学到的模型在(随机)测试上 ~0.5; 真实标签学到的能泛化
Xte_r, yte_r = gen(2000, 11, random_labels=True)
Xte_s, yte_s = gen(2000, 12, random_labels=False)
test_rand = np.mean(np.sign(Xte_r@wr) != yte_r)
test_real = np.mean(np.sign(Xte_s@ws) != yte_s)
print(f'测试误差: 随机标签模型={test_rand:.3f} (≈随机猜 0.5), 真实标签模型={test_real:.3f} (能泛化)')
assert test_real < test_rand, '真实标签模型泛化, 随机标签模型不泛化'
print('✅ 同一模型类: 随机标签时有效容量=最坏容量(不泛化); 真实标签时有效容量小(泛化)。')

## 2 · 归一化 margin 分布：真实标签 vs 随机标签

归一化 margin $\gamma_i = y_i(w^\top x_i)/\|w\|$。理论：拟合随机标签需要把权重拧大、margin 崩塌；真实标签的解 margin 更健康。对比两者的 margin 分布（中位数）。

In [ ]:
def norm_margins(X, y, w):
    return (y * (X @ w)) / np.linalg.norm(w)
m_real = norm_margins(Xs, ys, ws)
m_rand = norm_margins(Xr, yr, wr)
print(f'真实标签 margin: 中位数={np.median(m_real):.4f}, 最小={m_real.min():.4f}')
print(f'随机标签 margin: 中位数={np.median(m_rand):.4f}, 最小={m_rand.min():.4f}')
print(f'解范数: 真实||w||={np.linalg.norm(ws):.3f}, 随机||w||={np.linalg.norm(wr):.3f}')
# 插值解 margin 全为 1/||w|| (因为 y(Xw)=1 当 y=sign 且 Xw=y... 实际 y*Xw=|Xw|)
# 关键对比: 随机标签需要更大的范数来插值 -> margin/复杂度更差
assert np.linalg.norm(wr) > np.linalg.norm(ws), '拟合随机标签需要更大权重范数'
print('✅ 随机标签解范数更大 -> 范数/margin 复杂度更高 -> 范数依赖的界正确区分两者。')

## 3 · PAC-Bayes 界：反演 KL 数值计算（非平凡界）

PAC-Bayes: $\mathrm{kl}(\hat R_n\|R)\le B$，其中 $B=\frac{\mathrm{KL}(Q\|P)+\ln(2\sqrt n/\delta)}{n}$。同方差高斯的 $\mathrm{KL}(Q\|P)=\|\mu_Q-\mu_P\|^2/(2\sigma^2)$。用二分搜索**反演 KL** 求 $R$ 上界，并与 Pinsker 放缩对比（反演更紧）。

In [ ]:
def kl_bern(q, p):
    p = min(max(p, 1e-12), 1-1e-12); q = min(max(q, 1e-12), 1-1e-12)
    return q*np.log(q/p) + (1-q)*np.log((1-q)/(1-p))
def kl_gauss(muQ, muP, sigma):
    return np.sum((muQ - muP)**2) / (2*sigma**2)
def pac_bayes_bound(R_emp, KLqp, n, delta=0.05):
    B = (KLqp + np.log(2*np.sqrt(n)/delta)) / n
    # 反演 kl: 最大的 R 使 kl(R_emp || R) <= B
    lo, hi = R_emp, 1 - 1e-9
    for _ in range(100):
        mid = (lo + hi)/2
        if kl_bern(R_emp, mid) <= B: lo = mid
        else: hi = mid
    pinsker = R_emp + np.sqrt(B/2)         # Pinsker 放缩(更松)
    return lo, pinsker, B
muP = np.zeros(10); muQ = rng.standard_normal(10)*0.3; sigma = 1.0
KLqp = kl_gauss(muQ, muP, sigma); n_pb = 5000; R_emp = 0.05
R_bound, pinsker, B = pac_bayes_bound(R_emp, KLqp, n_pb)
print(f'KL(Q||P)={KLqp:.3f}, B={B:.4f}, 经验风险={R_emp}')
print(f'PAC-Bayes 反演KL 上界 R <= {R_bound:.4f}  (非平凡: <1 ✓)')
print(f'Pinsker 放缩上界     R <= {pinsker:.4f}  (更松)')
assert R_emp < R_bound < 1.0, '反演界应非平凡(在(R_emp,1)内)'
assert R_bound <= pinsker + 1e-9, '反演 KL 应比 Pinsker 放缩紧'
print('✅ PAC-Bayes 反演 KL 给出非平凡(<1)且比 Pinsker 紧的泛化界。')

In [ ]:
# 平坦极小 -> Q 可更宽(sigma 大) -> KL 小 -> 界更紧。验证 KL 随后验贴近先验而减小。
for shift in [0.1, 0.3, 0.6]:
    KL = kl_gauss(np.ones(10)*shift, np.zeros(10), 1.0)
    Rb, _, _ = pac_bayes_bound(0.05, KL, 5000)
    print(f'后验偏移={shift}: KL={KL:.3f} -> 界 R<={Rb:.4f}')
kl_small = kl_gauss(np.ones(10)*0.1, np.zeros(10), 1.0)
kl_big   = kl_gauss(np.ones(10)*0.6, np.zeros(10), 1.0)
assert pac_bayes_bound(0.05, kl_small, 5000)[0] < pac_bayes_bound(0.05, kl_big, 5000)[0]
print('✅ 后验越贴近先验(KL 小, 对应平坦解) -> 界越紧 —— 这是 Dziugaite&Roy 的核心。')

## 4 · 双下降：最小范数解在插值阈值处范数爆炸

扫过参数维度 $p$（固定 $n$），用最小范数解拟合含噪数据，画出**测试误差**与**解范数**。
理论：$p\approx n$（插值阈值）处测试误差冲高、解范数爆炸；$p\gg n$ 后回落（噪声摊薄）。

In [ ]:
def double_descent(n=40, noise=0.5, seed=0):
    g = np.random.default_rng(seed)
    d_full = 200; w_true = g.standard_normal(d_full); w_true /= np.linalg.norm(w_true)
    Xtr_full = g.standard_normal((n, d_full)); Xte_full = g.standard_normal((400, d_full))
    ytr = Xtr_full @ w_true + g.standard_normal(n)*noise   # 含噪回归目标
    yte = Xte_full @ w_true
    ps = [5, 15, 30, 38, 40, 42, 50, 80, 150, 200]
    test_err, norms = [], []
    for p in ps:
        Xtr, Xte = Xtr_full[:, :p], Xte_full[:, :p]
        if p <= n:   # 欠参数: 普通最小二乘
            w = np.linalg.lstsq(Xtr, ytr, rcond=None)[0]
        else:        # 过参数: 最小范数插值
            w = Xtr.T @ np.linalg.solve(Xtr @ Xtr.T + 1e-10*np.eye(n), ytr)
        test_err.append(np.mean((Xte @ w - yte)**2))
        norms.append(np.linalg.norm(w))
    return ps, test_err, norms
ps, te, nm = double_descent()
print(f'{"p":>5s} {"测试MSE":>10s} {"||w||":>10s}')
for p, e, nn in zip(ps, te, nm):
    mark = '  <- 插值阈值 p≈n' if p == 40 else ''
    print(f'{p:>5d} {e:>10.3f} {nn:>10.3f}{mark}')
# 峰值应在 p≈n=40 附近; 之后回落
peak = ps[int(np.argmax(te))]
print(f'\n测试误差峰值在 p={peak} (应≈n=40)')
assert 38 <= peak <= 50, '双下降峰值应在插值阈值 p≈n 附近'
assert te[-1] < max(te), 'p>>n 后测试误差应回落(第二次下降)'
assert nm[ps.index(40)] > nm[ps.index(200)], '解范数在阈值处大于 p>>n 处(范数回落)'
print('✅ 双下降: 测试误差在 p≈n 冲高、p>>n 回落; 解范数在阈值爆炸 —— 机制被钉死。')

---
## ✏️ 练习 1：随机标签拟合

实现 `can_fit_random(n, d, seed)` 返回过参数化线性模型拟合随机标签的训练误差（应为 0 当 $d\ge n$）。

In [ ]:
def can_fit_random(n, d, seed=0):
    # TODO: 生成 X(n,d), 随机 ±1 标签 y; 最小范数解 w=X^T(XX^T)^{-1}y;
    #       返回训练误差 mean(sign(Xw)!=y)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
err_over = can_fit_random(30, 60)     # 过参数化
assert err_over < 1e-9, '过参数化(d>=n)应零训练误差拟合随机标签'
print(f'd=60>n=30: 随机标签训练误差={err_over:.2e} (零)')
print('✅ 练习 1 通过：过参数化能记忆随机标签 -> 经典界失效')

## ✏️ 练习 2：归一化 margin

实现 `normalized_margins(X, y, w)` 返回归一化 margin 数组 $y_i(w^\top x_i)/\|w\|$。

In [ ]:
def normalized_margins(X, y, w):
    # TODO: 返回 (y * (X @ w)) / ||w||
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
Xt, yt = gen(20, 5, random_labels=False); wt = fit_interp(Xt, yt)
mg = normalized_margins(Xt, yt, wt)
assert mg.shape == (20,), 'margin 数每个样本一个'
# 缩放 w 不改变归一化 margin
assert np.allclose(normalized_margins(Xt, yt, 3*wt), mg), '归一化 margin 应对 w 缩放不变'
# 插值解上 margin = |Xw|/||w|| > 0 (全部分对)
assert np.all(mg > -1e-9), '插值解应所有 margin 非负'
print(f'归一化 margin 中位数={np.median(mg):.4f}, 对 w 缩放不变 ✓')
print('✅ 练习 2 通过：归一化 margin(尺度不变)是 margin 界的核心量')

## ✏️ 练习 3：PAC-Bayes 界

实现 `pac_bayes(R_emp, KLqp, n, delta)` 返回反演 KL 的 $R$ 上界（复用 worked 3 的 `kl_bern` 二分搜索逻辑）。

In [ ]:
def pac_bayes(R_emp, KLqp, n, delta=0.05):
    # TODO: B=(KLqp+ln(2 sqrt(n)/delta))/n; 二分搜索最大 R 使 kl_bern(R_emp,R)<=B
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
Rb = pac_bayes(0.1, 2.0, 10000)
assert 0.1 < Rb < 1.0, '应在 (R_emp, 1) 内'
# KL 越大界越松; n 越大界越紧
assert pac_bayes(0.1, 5.0, 10000) > pac_bayes(0.1, 1.0, 10000), 'KL 大 -> 界松'
assert pac_bayes(0.1, 2.0, 100000) < pac_bayes(0.1, 2.0, 10000), 'n 大 -> 界紧'
print(f'R_emp=0.1, KL=2, n=1e4 -> R<={Rb:.4f}')
print('✅ 练习 3 通过：PAC-Bayes 界随 KL 增大变松、随 n 增大变紧')

## ✏️ 练习 4：压缩界直觉 —— 有效参数 vs 间隙

压缩界：可压缩到 $k$ 个有效参数 → 间隙 $\sim\sqrt{k/n}$。实现 `compression_gap(k, n)` 返回 $\sqrt{k/n}$，并验证 $k$ 越小（压得越狠）间隙越小。

In [ ]:
def compression_gap(k, n):
    # TODO: 返回 sqrt(k/n)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
g_small = compression_gap(100, 50000); g_big = compression_gap(10000, 50000)
print(f'k=100 -> 间隙 {g_small:.4f};  k=10000 -> 间隙 {g_big:.4f}')
assert g_small < g_big, '有效参数越少(压缩越狠)间隙越小'
assert abs(compression_gap(200,50000)/compression_gap(50,50000) - 2.0) < 1e-9, 'k 增 4 倍间隙增 2 倍'
print('✅ 练习 4 通过：压缩界用有效参数 k(≪总参数)度量容量 -> 解释 Zhang 难题')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1
def can_fit_random(n, d, seed=0):
    g = np.random.default_rng(seed)
    X = g.standard_normal((n, d)); y = g.choice([-1., 1.], n)
    w = X.T @ np.linalg.solve(X @ X.T, y)
    return np.mean(np.sign(X @ w) != y)

In [ ]:
# 练习 2
def normalized_margins(X, y, w):
    return (y * (X @ w)) / np.linalg.norm(w)

In [ ]:
# 练习 3
def pac_bayes(R_emp, KLqp, n, delta=0.05):
    B = (KLqp + np.log(2*np.sqrt(n)/delta)) / n
    lo, hi = R_emp, 1 - 1e-9
    for _ in range(100):
        mid = (lo + hi)/2
        if kl_bern(R_emp, mid) <= B: lo = mid
        else: hi = mid
    return lo

In [ ]:
# 练习 4
def compression_gap(k, n):
    return np.sqrt(k / n)

---
## 🧪 真实数据胶囊：真实数据上随机标签 vs 真实标签的范数差距

用真实数据（sklearn digits 两类；失败回退到真实形状合成）验证：拟合**真实**标签的最小范数解的范数，显著**小于**拟合**随机**标签的——这是范数依赖的界能区分泛化好坏的真实数据证据。

In [ ]:
try:
    from sklearn.datasets import load_digits
    dig = load_digits(); m = np.isin(dig.target, [3, 8])
    Xd = dig.data[m].astype(float); yd = (dig.target[m] == 3).astype(float)*2 - 1
    Xd = (Xd - Xd.mean(0)) / (Xd.std(0) + 1e-9)
    src = 'sklearn digits 3vs8 (真实)'
except Exception as e:
    g = np.random.default_rng(0); Xd = g.standard_normal((360, 64))
    yd = np.sign(Xd @ (np.arange(64)<3)); src = f'回退合成: {type(e).__name__}'
# 取过参数化子集: 样本数 < 特征数
n_sub = 40; Xd = Xd[:n_sub]; yd = yd[:n_sub]
print(f'数据来源: {src}; 子集 {Xd.shape} (n={n_sub} < d={Xd.shape[1]}, 过参数化)')

In [ ]:
def min_norm_fit(X, y):
    return X.T @ np.linalg.solve(X @ X.T + 1e-8*np.eye(len(X)), y)
w_true_lab = min_norm_fit(Xd, yd)
y_rand = np.random.default_rng(7).choice([-1., 1.], len(yd))
w_rand_lab = min_norm_fit(Xd, y_rand)
print(f'真实标签最小范数解 ||w|| = {np.linalg.norm(w_true_lab):.3f}')
print(f'随机标签最小范数解 ||w|| = {np.linalg.norm(w_rand_lab):.3f}')
assert np.linalg.norm(w_rand_lab) > np.linalg.norm(w_true_lab), '随机标签需要更大范数'
print('✅ 真实数据: 拟合随机标签的范数 > 真实标签 -> 范数依赖的界正确区分。')

**🧪 胶囊练习**：实现 `norm_ratio()` 返回 `随机标签范数 / 真实标签范数`（应 >1，量化复杂度差距）。

In [ ]:
def norm_ratio():
    # TODO: 返回 ||w_rand_lab|| / ||w_true_lab||
    raise NotImplementedError

In [ ]:
# 胶囊自测
def norm_ratio():
    return np.linalg.norm(w_rand_lab) / np.linalg.norm(w_true_lab)
ratio = norm_ratio()
print(f'随机/真实 范数比 = {ratio:.3f} (>1)')
assert ratio > 1.0, '随机标签复杂度(范数)更高'
print('✅ 胶囊练习通过：随机标签范数比真实大 -> 有效容量差距的真实数据证据')

In [ ]:
# 📖 胶囊参考答案
def norm_ratio():
    return np.linalg.norm(w_rand_lab) / np.linalg.norm(w_true_lab)

---
### 小结（全课收口）
- Zhang 随机标签可拟合 -> 最坏容量巨大、经典界 vacuous; 但真实标签有效容量小 -> 泛化（同一模型类，已对拍）。
- margin 界：用范数/间隔替代参数计数; 随机标签需大范数 -> 界正确区分（已实测范数差距）。
- PAC-Bayes 反演 KL 界：非平凡(<1)且比 Pinsker 紧; 平坦解(KL 小) -> 界更紧（Dziugaite&Roy 的核心）。
- 压缩界/有效容量: margin、PAC-Bayes、压缩、平坦极小是同一个『有效容量小』的化身。
- 双下降: 最小范数解在 p≈n 范数爆炸、p>>n 回落（已复现峰值在阈值处）; benign overfitting 的谱条件。

**全课完。** 你现在有了研究科学家的理论脊梁：从『学习何时可能』(VC/Rademacher)、『优化能否找到解』(收敛率)、到『深度学习为何反常有效』(NTK/隐式偏置/现代界)——每一条都被你的数值实验亲手钉死。